# AMMS 302 — Week 10: Health Information Ethics & Regulations
**Data privacy · Security · PII · PDPA พ.ศ. 2562 · HIPAA · นโยบายคลาวด์กลาง**

> เปิดคู่กับ [สไลด์ wk10](./wk10.html) — Lab: audit PII + pseudonymize ข้อมูลผู้ป่วย

### 🎯 Learning objectives (CLO3/CLO4)
- จำแนก PII / sensitive data ในชุดข้อมูลสุขภาพได้
- อธิบายหลัก PDPA (ไทย) vs HIPAA (สากล) และ Safe Harbor de-identification
- ทำ pseudonymization (hash) + masking ด้วย pandas ได้จริง

### 📚 Official references
- **PDPC Thailand (คณะกรรมการคุ้มครองข้อมูลส่วนบุคคล):** [pdpc.or.th](https://pdpc.or.th/) — กฎหมาย PDPA, คู่มือประกอบ
- **HIPAA (US HHS):** [hhs.gov/hipaa](https://www.hhs.gov/hipaa/index.html) · [De-identification guidance](https://www.hhs.gov/hipaa/for-professionals/special-topics/de-identification/index.html) (Safe Harbor 18 identifiers)
- **Thai G-Cloud policy:** [cloud.dga.or.th](https://cloud.dga.or.th/) · [DGA](https://www.dga.or.th/) · ETDA: [etda.or.th](https://www.etda.or.th/)
- Python: [hashlib](https://docs.python.org/3/library/hashlib.html) · [pandas](https://pandas.pydata.org/docs/)

---


## §1 จำแนก PII ใน patients_data.csv (สไลด์ 03–04)
**PII (Personally Identifiable Information)** = ข้อมูลที่ชี้ตัวบุคคลได้ — ในเวชระเบียนถือเป็น *sensitive* ตาม PDPA §26


In [ ]:
import pandas as pd
df = pd.read_csv("patients_data.csv")
print(df.columns.tolist()); df.head(3)

In [ ]:
# §1.1 Audit: จัดประเภทแต่ละคอลัมน์
audit = {
 'subject_id':     ('Direct identifier', 'รหัสผู้ป่วย — ถือ direct ถ้าภายนอกระบบรู้ mapping'),
 'Name':           ('Direct identifier', 'ชื่อ-นามสกุล = PII ชัดเจน (PDPA/HIPAA)'),
 'Age':            ('Quasi-identifier', 'อายุร่วมกับอื่น → re-identification'),
 'gender':         ('Quasi-identifier', 'เพศ'),
 'systolic_bp':    ('Clinical value', 'ไม่ใช่ PII เอง — แต่ผูกกับ identity = sensitive'),
 'HbA1c_level':    ('Clinical value', 'เช่นเดียวกัน'),
 'admission_date': ('Quasi-identifier', 'วันที่ช่วย narrow down ตัวตน'),
}
display(pd.DataFrame(audit, index=['class','note']).T)
# 💡 HIPAA Safe Harbor 18 identifiers ดู: https://www.hhs.gov/hipaa/for-professionals/special-topics/de-identification/index.html

## §2 De-identification 3 ระดับ (สไลด์ 04)
| เทคนิค | ทำอะไร | ย้อนกลับได้? |
|---|---|---|
| **Masking** | ซ่อนบางส่วน `สม***` | ❌ |
| **Pseudonymization** | เปลี่ยน id → token (มี key แยก) | ⚠️ ได้ถ้ามี key |
| **Anonymization** | ลบ/แปลงจนชี้ตัวไม่ได้ | ❌ (Safe Harbor goal) |


In [ ]:
# §2.1 Masking ชื่อ — เก็บพระนาม/คำนำหน้า + ***
def mask_name(name):
    if not isinstance(name, str) or len(name) < 4:
        return "***"
    return name[:2] + "*" * max(1, len(name)-2)
df['Name_masked'] = df['Name'].map(mask_name)
display(df[['Name','Name_masked']].head(5))

# §2.2 Pseudonymize subject_id — SHA-256 + salt (deterministic: รันซ้ำได้ค่าเดิม)
import hashlib
SALT = b"AMMS302-class-salt"   # production: เก็บ salt แยกเป็นความลับ!
df['subject_hash'] = df['subject_id'].apply(
    lambda x: hashlib.sha256(SALT + str(x).encode()).hexdigest()[:12])
display(df[['subject_id','subject_hash']].head(5))
# 🔗 docs: https://docs.python.org/3/library/hashlib.html

## §3 Age generalization → ทางสู่ Safe Harbor (สไลด์ 05)
HIPAA Safe Harbor: อายุ > 89 ต้อง group เป็น "90+" · dates ต้องลบเฉพาะ year เท่านั้น — เรา generalize age เป็นช่วง 10 ปี (k-anonymity spirit)


In [ ]:
# §3.1 Generalize age → decade band; cap 89+ per HIPAA
age = pd.to_numeric(df['Age'], errors='coerce')
df['age_band'] = age.apply(lambda a: "89+" if pd.notna(a) and a >= 90 else
                                     ("unknown" if pd.notna(a)==False or a<0 else f"{int(a//10)*10}-{int(a//10)*10+9}"))
df['year_only'] = pd.to_datetime(df['admission_date'], errors='coerce').dt.year.astype('Int64')
display(df[['Age','age_band','admission_date','year_only']].head(6))

# §3.2 ตรวจ k-anonymity ง่าย ๆ: กลุ่ม (age_band, gender) ที่ n<5 = เสี่ยง
grp = df.groupby(['age_band','gender']).size().rename('k').reset_index()
display(grp)
print("⚠️ groups with k<5 (re-identification risk):")
display(grp[grp.k < 5])

## §4 Export ชุดข้อมูล safe-for-teaching (สไลด์ 06–08)
รวมทุกเทคนิค: drop Name, hash id, band age, year-only date — ผลลัพธ์คือ dataset ที่ปลอดภัยกว่าเดิมมาก (ยังไม่ใช่ certified anonymized!)


In [ ]:
safe = df[['subject_hash','gender','age_band','year_only','systolic_bp','HbA1c_level']].copy()
safe.to_csv('patients_deidentified.csv', index=False)
print(safe.shape); display(safe.head(5))
print("✅ saved patients_deidentified.csv — ใช้ใน classwork แทน raw ได้")
# จำ: PDPA ยังถือ hashed-id เป็น personal data ถ้า re-identify ได้ — นี่คือ pseudonymized, not anonymous

### ✅ Self-check
- audit ครบ 7 คอลัมน์ + จัด class ถูก
- mask/hash deterministic (รันซ้ำได้ค่าเดิม)
- age ≥90 → '89+' (HIPAA rule) · date → year only
- รู้ว่า k<5 คือ risk group

### 📝 Homework 10
1) เขียน 1 หน้า: เปรียบเทียบ PDPA vs HIPAA (3 ประเด็น: consent/rights/penalty) อ้างอิง [pdpc.or.th](https://pdpc.or.th/) และ [hhs.gov](https://www.hhs.gov/hipaa/index.html)
2) ส่ง `.ipynb` + `patients_deidentified.csv`

---
### 🔗 Specs รวม
[PDPC](https://pdpc.or.th/) · [HIPAA](https://www.hhs.gov/hipaa/index.html) · [Safe Harbor](https://www.hhs.gov/hipaa/for-professionals/special-topics/de-identification/index.html) · [G-Cloud](https://cloud.dga.or.th/) · [ETDA](https://www.etda.or.th/)
